# Encuentro 4 — Backend con FastAPI para el Agro
### Python, Datos e Ingeniería de IA Aplicada · UTN Rosario

**17 de junio de 2026 · 18:30–21:30**

---

## ¿Qué vamos a construir?

Una API REST para un sistema de **monitoreo inteligente de cultivos** que:

```
  Sensores IoT            FastAPI Backend              Clientes
  (campo)                 (este encuentro)        (apps, dashboards)
      │                          │                        │
      │  POST /sensores          │   GET /parcelas        │
      │─────────────────────────▶│◀───────────────────────│
      │                          │   GET /alertas         │
      │                          │◀───────────────────────│
      │                          │   POST /prediccion     │
      │                          │◀───────────────────────│
```

## Objetivos

| # | Objetivo |
|---|----------|
| 1 | Entender el protocolo HTTP y la arquitectura REST |
| 2 | Construir una API con FastAPI desde cero |
| 3 | Validar datos de entrada con Pydantic |
| 4 | Explorar la documentación automática (`/docs`) |
| 5 | Manejar errores correctamente y configurar logging |
| 6 | Aplicar **Vibe Engineering** para acelerar el desarrollo |

In [ ]:
# Instalación de dependencias
!pip install fastapi uvicorn httpx pydantic --quiet

---
## Parte 1 — El Protocolo HTTP

### ¿Qué es una API REST?

Una **API (Application Programming Interface)** es un contrato que permite que dos sistemas se comuniquen. **REST** es el estilo arquitectónico más usado en la web.

```
 Cliente                          Servidor
   │                                 │
   │  ──── REQUEST ───────────────▶  │
   │  método + URL + headers + body  │
   │                                 │
   │  ◀─── RESPONSE ──────────────   │
   │  status code + headers + body   │
   │                                 │
```

### Métodos HTTP

| Método | Uso en el agro | Ejemplo |
|--------|---------------|---------|
| `GET` | Consultar datos | `GET /parcelas/5` → info de la parcela 5 |
| `POST` | Crear / registrar | `POST /sensores` → registrar lectura de sensor |
| `PUT` | Reemplazar completamente | `PUT /parcelas/5` → actualizar toda la parcela |
| `PATCH` | Actualizar parcialmente | `PATCH /parcelas/5` → cambiar solo el cultivo |
| `DELETE` | Eliminar | `DELETE /parcelas/5` → borrar parcela |

### Status Codes más importantes

| Código | Significado | Cuándo usarlo |
|--------|-------------|---------------|
| `200 OK` | Todo bien | Respuesta exitosa general |
| `201 Created` | Recurso creado | Después de un POST exitoso |
| `400 Bad Request` | Error del cliente | Datos inválidos enviados |
| `404 Not Found` | No encontrado | Parcela/sensor inexistente |
| `422 Unprocessable Entity` | Error de validación | Pydantic rechazó los datos |
| `500 Internal Server Error` | Error del servidor | Bug en nuestro código |

In [ ]:
# Demostración: cómo se ve una request HTTP real
import httpx

# Hacemos un GET a una API pública de clima (sin key)
response = httpx.get("https://wttr.in/Rosario?format=j1", timeout=5.0)

print(f"Status code: {response.status_code}")
print(f"Content-Type: {response.headers.get('content-type')}")
print(f"Primeros 200 chars del body: {response.text[:200]}")

---
## Parte 2 — FastAPI desde Cero

### ¿Por qué FastAPI?

- **Rápido de escribir**: menos código que Flask o Django REST
- **Validación automática**: Pydantic integrado
- **Documentación automática**: Swagger UI y ReDoc sin configuración
- **Async nativo**: ideal para APIs de IA con modelos lentos
- **Tipado**: errores detectados antes de llegar a producción

### Anatomía de una app FastAPI

```python
from fastapi import FastAPI

app = FastAPI()          # ← la aplicación

@app.get("/ruta")        # ← decorador con método HTTP
def mi_funcion():        # ← función que maneja el request
    return {"clave": "valor"}  # ← respuesta automáticamente en JSON
```

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# ── La aplicación ──────────────────────────────────────────
app = FastAPI(
    title="API Monitoreo de Cultivos",
    description="Backend para seguimiento de parcelas agrícolas",
    version="1.0.0",
)

@app.get("/")
def inicio():
    return {"mensaje": "API de Monitoreo Agro funcionando ✓"}

@app.get("/salud")
def salud():
    return {"estado": "ok", "servicio": "monitoreo-cultivos"}

# ── Testear sin levantar servidor ─────────────────────────
client = TestClient(app)

res = client.get("/")
print(f"GET /  → {res.status_code}: {res.json()}")

res = client.get("/salud")
print(f"GET /salud → {res.status_code}: {res.json()}")

> **Para correr como servidor real**, usás el comando (en la terminal, no acá):
> ```bash
> uvicorn app_cultivos:app --reload
> ```
> `--reload` reinicia automáticamente cuando guardás el archivo. Ideal para desarrollo.
> Luego abrís `http://localhost:8000` en el navegador.

---
## Parte 3 — Rutas y Parámetros

### Tipos de parámetros en FastAPI

```
GET /parcelas/{id}?zona=norte&activa=true
         │              └── query params (opcionales)
         └── path param (obligatorio, parte de la URL)
```

| Tipo | Sintaxis Python | URL de ejemplo |
|------|----------------|----------------|
| Path param | `def f(id: int)` | `/parcelas/42` |
| Query param | `def f(zona: str = None)` | `/parcelas?zona=norte` |
| Body (JSON) | `def f(data: MiModelo)` | POST con JSON en el body |

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from typing import Optional

app = FastAPI()

# Base de datos simulada
parcelas_db = [
    {"id": 1, "nombre": "El Ombú",    "cultivo": "soja",  "zona": "norte", "activa": True},
    {"id": 2, "nombre": "La Cañada",  "cultivo": "maíz",  "zona": "sur",   "activa": True},
    {"id": 3, "nombre": "Don Ramón",  "cultivo": "trigo", "zona": "norte", "activa": False},
]

# PATH PARAM: /parcelas/1
@app.get("/parcelas/{parcela_id}")
def obtener_parcela(parcela_id: int):
    for p in parcelas_db:
        if p["id"] == parcela_id:
            return p
    return {"error": "parcela no encontrada"}

# QUERY PARAMS: /parcelas?zona=norte&activa=true
@app.get("/parcelas")
def listar_parcelas(zona: Optional[str] = None, activa: Optional[bool] = None):
    resultado = parcelas_db
    if zona:
        resultado = [p for p in resultado if p["zona"] == zona]
    if activa is not None:
        resultado = [p for p in resultado if p["activa"] == activa]
    return {"total": len(resultado), "parcelas": resultado}

# ── Tests ─────────────────────────────────────────────────
client = TestClient(app)

print("Path param → id=2:")
print(client.get("/parcelas/2").json())

print("\nQuery param → zona=norte, activa=true:")
print(client.get("/parcelas?zona=norte&activa=true").json())

---
## Parte 4 — Validación de Datos con Pydantic

### ¿Por qué validar los datos?

En el agro, los sensores pueden enviar:
- Valores fuera de rango (`temperatura: -999.0`)
- Campos faltantes (`{"parcela_id": 5}` sin valor del sensor)
- Tipos incorrectos (`{"humedad": "mucha"}` en lugar de un número)

Pydantic intercepta todo esto **antes** de que llegue a tu lógica de negocio.

### BaseModel básico

```python
from pydantic import BaseModel, Field

class LecturaSensor(BaseModel):
    parcela_id: int
    tipo: str
    valor: float = Field(..., ge=-50, le=100)  # entre -50 y 100
    unidad: str = "°C"                          # valor por defecto
```

`Field(...)` significa **obligatorio**. Los demás son opcionales si tienen default.

In [ ]:
from pydantic import BaseModel, Field, validator
from typing import Optional, Literal
from datetime import datetime

# ── Modelos del dominio agro ───────────────────────────────

class Parcela(BaseModel):
    nombre: str = Field(..., min_length=2, max_length=100)
    cultivo: str
    superficie_ha: float = Field(..., gt=0, description="Hectáreas")
    zona: Literal["norte", "sur", "este", "oeste"]
    activa: bool = True

class LecturaSensor(BaseModel):
    parcela_id: int
    tipo: Literal["temperatura", "humedad_suelo", "ph", "lluvia"]
    valor: float
    unidad: str
    timestamp: Optional[datetime] = None

    @validator("valor")
    def validar_rango(cls, v, values):
        rangos = {
            "temperatura": (-20, 60),
            "humedad_suelo": (0, 100),
            "ph": (0, 14),
            "lluvia": (0, 500),
        }
        tipo = values.get("tipo")
        if tipo in rangos:
            min_v, max_v = rangos[tipo]
            if not (min_v <= v <= max_v):
                raise ValueError(f"{tipo} debe estar entre {min_v} y {max_v}, recibido: {v}")
        return v

# ── Probar validaciones ────────────────────────────────────

# Válido
lectura = LecturaSensor(
    parcela_id=1,
    tipo="temperatura",
    valor=24.5,
    unidad="°C"
)
print("Lectura válida:", lectura.model_dump())

# Inválido → Pydantic lanza ValidationError
try:
    LecturaSensor(parcela_id=1, tipo="temperatura", valor=999, unidad="°C")
except Exception as e:
    print("\nError de validación:", e)

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, validator
from typing import Optional, Literal
from datetime import datetime

app = FastAPI(title="API Cultivos")

# Almacenamiento en memoria
parcelas_db: dict = {}
lecturas_db: list = []
next_id = {"parcela": 1}

class ParcelaIn(BaseModel):
    nombre: str = Field(..., min_length=2)
    cultivo: str
    superficie_ha: float = Field(..., gt=0)
    zona: Literal["norte", "sur", "este", "oeste"]

class LecturaSensorIn(BaseModel):
    parcela_id: int
    tipo: Literal["temperatura", "humedad_suelo", "ph", "lluvia"]
    valor: float
    unidad: str

    @validator("valor")
    def rango_valido(cls, v, values):
        rangos = {"temperatura": (-20, 60), "humedad_suelo": (0, 100), "ph": (0, 14)}
        tipo = values.get("tipo")
        if tipo in rangos:
            lo, hi = rangos[tipo]
            if not (lo <= v <= hi):
                raise ValueError(f"{tipo} fuera de rango [{lo}, {hi}]")
        return v

@app.post("/parcelas", status_code=201)
def crear_parcela(parcela: ParcelaIn):
    id_nuevo = next_id["parcela"]
    parcelas_db[id_nuevo] = {"id": id_nuevo, **parcela.model_dump()}
    next_id["parcela"] += 1
    return parcelas_db[id_nuevo]

@app.post("/sensores", status_code=201)
def registrar_lectura(lectura: LecturaSensorIn):
    if lectura.parcela_id not in parcelas_db:
        raise HTTPException(status_code=404, detail=f"Parcela {lectura.parcela_id} no encontrada")
    registro = {**lectura.model_dump(), "timestamp": datetime.now().isoformat()}
    lecturas_db.append(registro)
    return registro

# ── Tests ─────────────────────────────────────────────────
client = TestClient(app)

# Crear parcela
r = client.post("/parcelas", json={
    "nombre": "El Ombú", "cultivo": "soja",
    "superficie_ha": 250.0, "zona": "norte"
})
print("POST /parcelas:", r.status_code, r.json())

# Registrar lectura válida
r = client.post("/sensores", json={
    "parcela_id": 1, "tipo": "temperatura",
    "valor": 28.3, "unidad": "°C"
})
print("POST /sensores:", r.status_code, r.json())

# Lectura inválida (pH imposible)
r = client.post("/sensores", json={
    "parcela_id": 1, "tipo": "ph",
    "valor": 99.0, "unidad": "pH"
})
print("POST /sensores inválido:", r.status_code, r.json()["detail"])

---
## Parte 5 — Documentación Automática

FastAPI genera documentación **sin que escribas nada extra**.

Cuando levantás el servidor con `uvicorn app_cultivos:app --reload`:

| URL | Qué es |
|-----|--------|
| `http://localhost:8000/docs` | **Swagger UI** — interfaz interactiva para probar endpoints |
| `http://localhost:8000/redoc` | **ReDoc** — documentación más legible |
| `http://localhost:8000/openapi.json` | Schema OpenAPI en JSON (para integraciones) |

### Enriquecer la documentación con metadatos

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="API de Monitoreo de Cultivos",
    description="""
## Sistema de Monitoreo Inteligente Agropecuario

Permite:
- Gestionar **parcelas** y sus cultivos
- Registrar **lecturas de sensores** (temperatura, humedad, pH)
- Obtener **predicciones de IA** sobre estado de los cultivos
- Consultar **alertas** generadas automáticamente
    """,
    version="1.0.0",
    contact={"name": "Kevin", "email": "kevin@utn.edu.ar"},
)

class Parcela(BaseModel):
    nombre: str = Field(..., example="El Ombú", description="Nombre identificatorio de la parcela")
    cultivo: str = Field(..., example="soja", description="Tipo de cultivo actual")
    superficie_ha: float = Field(..., example=250.0, description="Superficie en hectáreas")

@app.get(
    "/parcelas/{id}",
    summary="Obtener parcela por ID",
    description="Retorna la información completa de una parcela dado su identificador numérico.",
    response_description="Datos de la parcela",
    tags=["Parcelas"],
)
def obtener_parcela(id: int):
    """Busca una parcela por su ID. Retorna 404 si no existe."""
    return {"id": id, "nombre": "El Ombú", "cultivo": "soja"}

# Ver el schema generado
from fastapi.testclient import TestClient
client = TestClient(app)
schema = client.get("/openapi.json").json()
print("Título:", schema["info"]["title"])
print("Endpoints:", list(schema["paths"].keys()))

---
## Parte 6 — Manejo de Errores

### HTTPException — el error controlado

```python
raise HTTPException(status_code=404, detail="Parcela no encontrada")
```

### Errores comunes en APIs agro

| Situación | Status code | Ejemplo |
|-----------|-------------|------|
| Parcela no existe | 404 | `GET /parcelas/999` |
| Datos de sensor inválidos | 422 | `valor: "mucha"` |
| Falta campo obligatorio | 422 | Body sin `parcela_id` |
| Error en modelo de IA | 500 | Excepción no manejada |
| Usuario sin permisos | 403 | (para cuando agregues auth) |

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

app = FastAPI()

parcelas_db = {
    1: {"id": 1, "nombre": "El Ombú",   "cultivo": "soja"},
    2: {"id": 2, "nombre": "La Cañada", "cultivo": "maíz"},
}

# ── Error controlado con HTTPException ────────────────────
@app.get("/parcelas/{parcela_id}")
def obtener_parcela(parcela_id: int):
    if parcela_id not in parcelas_db:
        raise HTTPException(
            status_code=404,
            detail={
                "error": "parcela_no_encontrada",
                "mensaje": f"No existe la parcela con ID {parcela_id}",
                "sugerencia": "Verificá el ID o consultá GET /parcelas para ver las disponibles",
            }
        )
    return parcelas_db[parcela_id]

# ── Handler global para errores no esperados ──────────────
@app.exception_handler(Exception)
async def handler_general(request: Request, exc: Exception):
    return JSONResponse(
        status_code=500,
        content={"error": "error_interno", "detalle": str(exc)},
    )

# ── Tests ─────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=False)

res = client.get("/parcelas/1")
print(f"Existe → {res.status_code}: {res.json()}")

res = client.get("/parcelas/999")
print(f"No existe → {res.status_code}: {res.json()}")

---
## Parte 7 — Logging y Depuración

### ¿Por qué logging en el agro?

Cuando un sensor registra un valor anómalo o una predicción falla en el campo, necesitás saber:
- ¿Qué parcela envió el dato?
- ¿En qué momento?
- ¿Qué valor llegó exactamente?
- ¿Qué hizo el sistema?

Eso es el logging: el registro de lo que pasó.

### Niveles de log

| Nivel | Cuándo usarlo |
|-------|---------------|
| `DEBUG` | Detalles internos (desarrollo) |
| `INFO` | Eventos normales (producción) |
| `WARNING` | Algo inusual pero no crítico |
| `ERROR` | Error que impide completar la operación |
| `CRITICAL` | Fallo del sistema |

> **Regla práctica:** en producción usá `INFO`. En desarrollo usá `DEBUG`.

In [ ]:
import logging
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("api.cultivos")

app = FastAPI()

class LecturaIn(BaseModel):
    parcela_id: int
    tipo: str
    valor: float

UMBRALES_CRITICOS = {"temperatura": 45.0, "ph": 4.5}

@app.post("/sensores")
def registrar_lectura(lectura: LecturaIn):
    logger.info("Lectura recibida | parcela=%d tipo=%s valor=%.2f",
                lectura.parcela_id, lectura.tipo, lectura.valor)

    # Verificar umbral crítico
    if lectura.tipo in UMBRALES_CRITICOS:
        if lectura.valor >= UMBRALES_CRITICOS[lectura.tipo]:
            logger.warning(
                "ALERTA UMBRAL | parcela=%d tipo=%s valor=%.2f umbral=%.2f",
                lectura.parcela_id, lectura.tipo,
                lectura.valor, UMBRALES_CRITICOS[lectura.tipo]
            )

    return {"registrada": True, "lectura": lectura.model_dump()}

client = TestClient(app)

print("=== Lectura normal ===")
client.post("/sensores", json={"parcela_id": 1, "tipo": "temperatura", "valor": 28.0})

print("\n=== Lectura crítica ===")
client.post("/sensores", json={"parcela_id": 3, "tipo": "temperatura", "valor": 47.5})

---
## Parte 8 — Vibe Engineering

> **Vibe Engineering** es programar en colaboración con IA: describís lo que querés construir y la IA genera el scaffolding. Vos revisás, ajustás y entendés.

### El flujo

```
1. Describir claramente qué necesitás
       ↓
2. IA genera el código
       ↓
3. Vos revisás línea por línea (¿tiene sentido?)
       ↓
4. Ajustás lo que no encaja con tu dominio
       ↓
5. Probás con TestClient antes de integrar
```

### Prompt efectivo para FastAPI

```
Creame un endpoint POST /alertas en FastAPI que:
- Reciba un body con: parcela_id (int), tipo (str), nivel (info/warning/crítico), mensaje (str)
- Valide que el nivel sea uno de los tres valores
- Guarde la alerta en una lista en memoria
- Devuelva la alerta con un campo `id` autogenerado y `timestamp` actual
- Use HTTPException 404 si la parcela_id no está en un dict llamado `parcelas_db`
```

### Lo que debés revisar siempre en código IA

| Checkeo | Por qué |
|---------|---------|
| ¿Los tipos son correctos? | IA a veces confunde `str` con `int` |
| ¿La validación tiene sentido para el dominio? | Rangos, enums, obligatorios |
| ¿Los status codes son correctos? | 201 para crear, 200 para consultar |
| ¿El manejo de errores es explícito? | No `except Exception: pass` |
| ¿La lógica de negocio está en el endpoint o separada? | Preferí funciones separadas |

In [ ]:
# Código generado por IA + revisado con criterio de ingeniería
# Prompt: "endpoint para registrar alertas en el sistema de monitoreo agro"

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime

app = FastAPI()

# Estado en memoria (en producción: base de datos)
parcelas_db: dict = {1: "El Ombú", 2: "La Cañada"}
alertas_db: list = []
_next_alerta_id = 1

class AlertaIn(BaseModel):
    parcela_id: int
    tipo: str = Field(..., example="temperatura_crítica")
    nivel: Literal["info", "warning", "crítico"]
    mensaje: str = Field(..., min_length=5)

class AlertaOut(BaseModel):
    id: int
    parcela_id: int
    tipo: str
    nivel: str
    mensaje: str
    timestamp: str
    resuelta: bool

@app.post("/alertas", response_model=AlertaOut, status_code=201)
def crear_alerta(alerta: AlertaIn):
    global _next_alerta_id

    if alerta.parcela_id not in parcelas_db:
        raise HTTPException(
            status_code=404,
            detail=f"Parcela {alerta.parcela_id} no encontrada"
        )

    nueva = {
        "id": _next_alerta_id,
        "parcela_id": alerta.parcela_id,
        "tipo": alerta.tipo,
        "nivel": alerta.nivel,
        "mensaje": alerta.mensaje,
        "timestamp": datetime.now().isoformat(),
        "resuelta": False,
    }
    alertas_db.append(nueva)
    _next_alerta_id += 1
    return nueva

@app.get("/alertas")
def listar_alertas(nivel: str = None, resuelta: bool = None):
    resultado = alertas_db
    if nivel:
        resultado = [a for a in resultado if a["nivel"] == nivel]
    if resuelta is not None:
        resultado = [a for a in resultado if a["resuelta"] == resuelta]
    return {"total": len(resultado), "alertas": resultado}

# ── Tests ─────────────────────────────────────────────────
client = TestClient(app)

# Crear alerta crítica
r = client.post("/alertas", json={
    "parcela_id": 1,
    "tipo": "temperatura_critica",
    "nivel": "crítico",
    "mensaje": "Temperatura supera 45°C en zona norte"
})
print("Alerta creada:", r.status_code, r.json())

# Filtrar por nivel
r = client.get("/alertas?nivel=crítico")
print("Alertas críticas:", r.json())

---
## Parte 9 — Proyecto Integrador: API de Monitoreo de Cultivos

Juntamos todo lo visto en una API completa que podría desplegarse en Railway o cualquier servidor.

### Arquitectura

```
api_cultivos/
├── app.py          ← FastAPI app + startup
├── models.py       ← Modelos Pydantic
├── routes/
│   ├── parcelas.py ← Endpoints de parcelas
│   ├── sensores.py ← Endpoints de sensores
│   └── alertas.py  ← Endpoints de alertas
└── requirements.txt
```

> En este notebook integramos todo en un solo archivo por claridad. El archivo `encuentro4/app_cultivos.py` tiene la versión completa lista para correr.

In [ ]:
import logging
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, validator
from typing import Literal, Optional
from datetime import datetime

# ── Logging ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("api.cultivos")

# ── Modelos ───────────────────────────────────────────────
class ParcelaIn(BaseModel):
    nombre: str = Field(..., min_length=2, example="El Ombú")
    cultivo: str = Field(..., example="soja")
    superficie_ha: float = Field(..., gt=0, example=250.0)
    zona: Literal["norte", "sur", "este", "oeste"]

class LecturaSensorIn(BaseModel):
    parcela_id: int
    tipo: Literal["temperatura", "humedad_suelo", "ph", "lluvia"]
    valor: float
    unidad: str

    @validator("valor")
    def rango_valido(cls, v, values):
        rangos = {"temperatura": (-20, 60), "humedad_suelo": (0, 100), "ph": (0, 14), "lluvia": (0, 500)}
        tipo = values.get("tipo")
        if tipo in rangos:
            lo, hi = rangos[tipo]
            if not lo <= v <= hi:
                raise ValueError(f"{tipo} debe estar en [{lo}, {hi}], recibido {v}")
        return v

class PrediccionIn(BaseModel):
    parcela_id: int
    modelo: Literal["stress_hidrico", "riesgo_helada", "rendimiento"]

# ── Estado en memoria ─────────────────────────────────────
parcelas: dict = {}
lecturas: list = []
alertas: list = []
_ids = {"parcela": 1, "alerta": 1}

UMBRALES = {
    "temperatura":    {"warning": 38.0, "crítico": 45.0},
    "humedad_suelo":  {"warning": 20.0, "crítico": 10.0},
    "ph":             {"warning": 5.0,  "crítico": 4.0},
}

# ── App ───────────────────────────────────────────────────
app = FastAPI(
    title="API Monitoreo de Cultivos",
    description="Sistema inteligente de seguimiento agropecuario",
    version="1.0.0",
)

@app.exception_handler(Exception)
async def handler_global(request: Request, exc: Exception):
    logger.error("Error no manejado: %s", exc)
    return JSONResponse(status_code=500, content={"error": "error_interno", "detalle": str(exc)})

# ── Parcelas ──────────────────────────────────────────────
@app.post("/parcelas", status_code=201, tags=["Parcelas"])
def crear_parcela(p: ParcelaIn):
    id_p = _ids["parcela"]
    parcelas[id_p] = {"id": id_p, **p.model_dump()}
    _ids["parcela"] += 1
    logger.info("Parcela creada | id=%d nombre=%s", id_p, p.nombre)
    return parcelas[id_p]

@app.get("/parcelas", tags=["Parcelas"])
def listar_parcelas(zona: Optional[str] = None):
    result = list(parcelas.values())
    if zona:
        result = [p for p in result if p["zona"] == zona]
    return {"total": len(result), "parcelas": result}

@app.get("/parcelas/{parcela_id}", tags=["Parcelas"])
def obtener_parcela(parcela_id: int):
    if parcela_id not in parcelas:
        raise HTTPException(404, detail=f"Parcela {parcela_id} no encontrada")
    return parcelas[parcela_id]

# ── Sensores ──────────────────────────────────────────────
@app.post("/sensores", status_code=201, tags=["Sensores"])
def registrar_lectura(lectura: LecturaSensorIn):
    if lectura.parcela_id not in parcelas:
        raise HTTPException(404, detail=f"Parcela {lectura.parcela_id} no encontrada")

    registro = {**lectura.model_dump(), "timestamp": datetime.now().isoformat()}
    lecturas.append(registro)
    logger.info("Lectura | parcela=%d tipo=%s valor=%.2f",
                lectura.parcela_id, lectura.tipo, lectura.valor)

    # Generar alerta si supera umbral
    if lectura.tipo in UMBRALES:
        for nivel, umbral in UMBRALES[lectura.tipo].items():
            if lectura.valor <= umbral if lectura.tipo in ["humedad_suelo", "ph"] else lectura.valor >= umbral:
                id_a = _ids["alerta"]
                alerta = {
                    "id": id_a,
                    "parcela_id": lectura.parcela_id,
                    "tipo": f"{lectura.tipo}_fuera_de_rango",
                    "nivel": nivel,
                    "mensaje": f"{lectura.tipo} = {lectura.valor} {lectura.unidad} (umbral {nivel}: {umbral})",
                    "timestamp": datetime.now().isoformat(),
                    "resuelta": False,
                }
                alertas.append(alerta)
                _ids["alerta"] += 1
                logger.warning("ALERTA %s | %s", nivel.upper(), alerta["mensaje"])
                break

    return registro

@app.get("/sensores", tags=["Sensores"])
def listar_lecturas(parcela_id: Optional[int] = None, tipo: Optional[str] = None):
    result = lecturas
    if parcela_id:
        result = [l for l in result if l["parcela_id"] == parcela_id]
    if tipo:
        result = [l for l in result if l["tipo"] == tipo]
    return {"total": len(result), "lecturas": result}

# ── Alertas ───────────────────────────────────────────────
@app.get("/alertas", tags=["Alertas"])
def listar_alertas(nivel: Optional[str] = None, resuelta: Optional[bool] = None):
    result = alertas
    if nivel:
        result = [a for a in result if a["nivel"] == nivel]
    if resuelta is not None:
        result = [a for a in result if a["resuelta"] == resuelta]
    return {"total": len(result), "alertas": result}

@app.patch("/alertas/{alerta_id}/resolver", tags=["Alertas"])
def resolver_alerta(alerta_id: int):
    for a in alertas:
        if a["id"] == alerta_id:
            a["resuelta"] = True
            logger.info("Alerta resuelta | id=%d", alerta_id)
            return a
    raise HTTPException(404, detail=f"Alerta {alerta_id} no encontrada")

# ── Predicciones (IA simulada) ────────────────────────────
@app.post("/prediccion", tags=["IA"])
def predecir(req: PrediccionIn):
    if req.parcela_id not in parcelas:
        raise HTTPException(404, detail=f"Parcela {req.parcela_id} no encontrada")

    # Buscar últimas lecturas de la parcela
    ultimas = [l for l in lecturas if l["parcela_id"] == req.parcela_id]
    if not ultimas:
        raise HTTPException(400, detail="No hay lecturas de sensores para esta parcela")

    # Simulación del modelo de IA
    resultados_simulados = {
        "stress_hidrico":  {"riesgo": "bajo",   "confianza": 0.85, "recomendacion": "Sin acción requerida"},
        "riesgo_helada":   {"riesgo": "medio",  "confianza": 0.72, "recomendacion": "Monitorear temperatura nocturna"},
        "rendimiento":     {"estimado_tn_ha": 3.8, "confianza": 0.68, "recomendacion": "Condiciones favorables"},
    }

    return {
        "parcela_id": req.parcela_id,
        "modelo": req.modelo,
        "resultado": resultados_simulados[req.modelo],
        "lecturas_analizadas": len(ultimas),
        "timestamp": datetime.now().isoformat(),
    }

In [ ]:
# ── Suite de integración completa ─────────────────────────
client = TestClient(app)

print("=" * 55)
print("TEST 1: Crear parcelas")
print("=" * 55)
for datos in [
    {"nombre": "El Ombú",    "cultivo": "soja",  "superficie_ha": 250.0, "zona": "norte"},
    {"nombre": "La Cañada",  "cultivo": "maíz",  "superficie_ha": 180.0, "zona": "sur"},
]:
    r = client.post("/parcelas", json=datos)
    print(f"  {r.status_code}: {r.json()['nombre']} (id={r.json()['id']})")

print("\n" + "=" * 55)
print("TEST 2: Registrar lecturas (incluye una crítica)")
print("=" * 55)
lecturas_test = [
    {"parcela_id": 1, "tipo": "temperatura",   "valor": 28.5,  "unidad": "°C"},
    {"parcela_id": 1, "tipo": "humedad_suelo",  "valor": 65.0,  "unidad": "%"},
    {"parcela_id": 1, "tipo": "temperatura",   "valor": 47.2,  "unidad": "°C"},  # ← crítico
    {"parcela_id": 2, "tipo": "ph",             "valor": 6.5,   "unidad": "pH"},
]
for l in lecturas_test:
    r = client.post("/sensores", json=l)
    print(f"  {r.status_code}: {l['tipo']}={l['valor']} en parcela {l['parcela_id']}")

print("\n" + "=" * 55)
print("TEST 3: Consultar alertas generadas")
print("=" * 55)
r = client.get("/alertas")
data = r.json()
print(f"  Total de alertas: {data['total']}")
for a in data["alertas"]:
    print(f"  [{a['nivel'].upper()}] {a['mensaje']}")

print("\n" + "=" * 55)
print("TEST 4: Predicción de IA")
print("=" * 55)
for modelo in ["stress_hidrico", "riesgo_helada", "rendimiento"]:
    r = client.post("/prediccion", json={"parcela_id": 1, "modelo": modelo})
    resultado = r.json()["resultado"]
    print(f"  {modelo}: {resultado}")

print("\n" + "=" * 55)
print("TEST 5: Resolver alerta")
print("=" * 55)
r = client.patch("/alertas/1/resolver")
print(f"  {r.status_code}: alerta resuelta = {r.json().get('resuelta')}")

---
## Resumen del Encuentro

```
✅  HTTP: métodos, status codes, requests/responses
✅  FastAPI: rutas, path params, query params, POST con body  
✅  Pydantic: validación de tipos, rangos, enums, validators
✅  Documentación automática: /docs y /redoc
✅  Manejo de errores: HTTPException + handler global
✅  Logging: niveles, formato, alertas por umbral
✅  Vibe Engineering: flujo con IA + revisión crítica
✅  Proyecto: API de monitoreo de cultivos completa
```

## Para continuar

```bash
# Correr la app del encuentro
cd encuentro4
uvicorn app_cultivos:app --reload
# Abrir http://localhost:8000/docs
```

## Ejercicios para practicar

1. **Agregar campo `lat/lon`** a `ParcelaIn` para geolocalización de la parcela
2. **Endpoint `GET /parcelas/{id}/resumen`** que retorne las últimas 5 lecturas de cada tipo de sensor
3. **Umbral configurable**: endpoint `POST /umbrales` que permita cambiar los umbrales de alerta en runtime
4. **Vibe**: pedile a una IA que agregue un endpoint `GET /dashboard` con estadísticas globales y revisá el código generado

---

> **Próximo encuentro (24/06):** con esta base, integraremos modelos de IA reales como embeddings, clasificadores de imágenes de cultivos y RAG sobre información agronómica.